<a href="https://colab.research.google.com/github/Thisis4oF/ComfyUIonColab/blob/main/ComfyUIonColab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ⚡ ปุ่มรันด่วนขั้นตอนทั้งหมดในคลิกเดียว (Master Auto-Launcher)
กรอกข้อมูลด้านล่างนี้แล้วกดปุ่ม Run เพียงครั้งเดียว ระบบจะจัดการเชื่อมโยงไดรฟ์ อัปเดต และเปิดเซิร์ฟเวอร์ ComfyUI ให้พร้อมใช้งานทันทีครับ

---

### 💡 หากคุณเข้าไปดูใน Google Drive บนเบราว์เซอร์/มือถือแล้วไม่พบโฟลเดอร์:
1. **เป็นเรื่องปกติของระบบซิงค์ไฟล์ใน Google Colab:** ตัวระบบจริงได้สร้างโฟลเดอร์เสร็จสิ้นและเขียนข้อมูลลงไปยังดิสก์เสมือนจริงของ Google Drive เรียบร้อยแล้ว แต่ API ฝั่งหน้าเว็บอาจยังประมวลผลอัปเดตหน้าจอไม่ทัน (ดีเลย์ 1-5 นาที)
2. **วิธีบังคับแสดงไฟล์แบบเร่งด่วน:**
   * ที่แถบเมนูด้านซ้ายของ Google Colab ให้คลิกไอคอน **แฟ้ม (Files)** -> กดปุ่ม **Refresh** (ไอคอนวงกลมหมุนวน)
   * ไปที่หน้าเว็บ Google Drive ของคุณ แล้วคลิกขวาที่พื้นที่ว่าง เลือก **"ตรวจสอบอัปเดตไฟล์ล่าสุด"** หรือทำกาการรีเฟรชหน้าเว็บ (F5)
   * ในปุ่มควบคุมด้านล่างนี้ มีระบบบังคับประสานเวลาและส่งข้อมูล `drive.flush_and_unmount()` ทำงานเรียบร้อยแล้วเมื่อมีการดาวน์โหลดเสร็จสิ้น

In [ ]:
#@title 🚀 กดเริ่มทำงานทั้งหมดแบบอัตโนมัติที่นี่
import os
import shutil
import subprocess
import threading
import time
import socket

# --- 1. ตรวจสอบและเชื่อมต่อ Google Drive แบบปลอดภัย ---
print("🔄 1. กำลังตรวจสอบการเชื่อมต่อ Google Drive...")
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    try:
        drive.mount('/content/drive', force_remount=False)
        print("✅ เชื่อมต่อ Google Drive สำเร็จ!")
    except Exception as e:
        print("⚠️ พยายามเชื่อมต่อแบบปกติล้มเหลว กำลังลองบังคับรีเมานต์:", str(e))
        drive.mount('/content/drive', force_remount=True)
else:
    print("✅ Google Drive เชื่อมต่ออยู่แล้ว ข้ามขั้นตอนนี้ได้เลย")

# --- 2. ตั้งค่าไดเรกทอรีการทำงานและตรวจสอบความถูกต้อง ---
DRIVE_PATH = "/content/drive/MyDrive/ComfyUI_Drive"
WORKSPACE = DRIVE_PATH + "/ComfyUI"

# หากโฟลเดอร์มีอยู่จริงแต่ไม่มีไฟล์สำคัญหลัก ให้มองว่าพังแล้วจัดการใหม่
is_corrupted = False
if os.path.exists(WORKSPACE):
    if not os.path.exists(os.path.join(WORKSPACE, "main.py")) or not os.path.exists(os.path.join(WORKSPACE, ".git")):
        print("⚠️ ตรวจพบโฟลเดอร์ ComfyUI เก่าเสียหายหรือไฟล์ไม่ครบถ้วน กำลังลบโฟลเดอร์ที่ขัดข้องเพื่อติดตั้งใหม่ให้สมบูรณ์...")
        try:
            shutil.rmtree(WORKSPACE)
        except:
            # หากลบผ่าน python แบบดั้งเดิมติดสิทธิ์ ให้ลบผ่านระบบ shell
            !rm -rf "{WORKSPACE}"
        is_corrupted = True

# ตรวจสอบว่าพาร์ทไหนใช้งานได้จริงหลังจากซ่อมแซม
possible_paths = [
    "/content/drive/MyDrive/ComfyUI_Drive/ComfyUI",
    "/content/drive/MyDrive/ComfyUI",
    "/content/ComfyUI"
]

active_workspace = WORKSPACE
found_valid = False
for path in possible_paths:
    if os.path.exists(os.path.join(path, "main.py")):
        active_workspace = path
        found_valid = True
        break

if not found_valid:
    print("📥 ไม่พบโฟลเดอร์โปรแกรม ComfyUI กำลังเริ่มต้นสร้างที่:", WORKSPACE)
    os.makedirs(DRIVE_PATH, exist_ok=True)
    os.chdir(DRIVE_PATH)
    !git clone https://github.com/comfyanonymous/ComfyUI
    active_workspace = WORKSPACE

print(f"📌 โฟลเดอร์ทำงานปัจจุบัน: {active_workspace}")
os.chdir(active_workspace)

# --- 3. อัปเดตซอร์สโค้ด ---
try:
    print("🔄 2. กำลังดึงอัปเดต ComfyUI ล่าสุด...")
    !git pull
except Exception as e:
    print("⚠️ ไม่สามารถดึงอัปเดตผ่าน git ได้:", str(e))

# บังคับซิงค์ไฟล์เข้าหน้าเว็บบน Google Drive ทันที
try:
    from google.colab import drive
    drive.flush_and_unmount()
    drive.mount('/content/drive', force_remount=False)
    print("🔄 ทำการซิงค์ไฟล์ประสานระบบกับเซิร์ฟเวอร์ Google Drive สำเร็จ!")
except Exception as e:
    pass

# --- 4. ติดตั้ง Dependencies และตัวเสริม ---
print("🔄 3. กำลังติดตั้งไลบรารี gguf และแพ็กเกจระบบเสริม...")
!pip install -q gguf
if os.path.exists("requirements.txt"):
    !pip install -q xformers!=0.0.18 -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu121
else:
    !pip install -q xformers!=0.0.18 torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu121

# ติดตั้ง Cloudflared
if not os.path.exists("/usr/local/bin/cloudflared"):
    print("🔄 4. กำลังดาวน์โหลดและติดตั้งตัวแปลงระบบเน็ตเวิร์ก Cloudflared...")
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
    !dpkg -i cloudflared-linux-amd64.deb > /dev/null

# --- 5. เริ่มต้นระบบจัดการคิวเซิร์ฟเวอร์ Cloudflare Tunnel ---
def iframe_thread(port):
    while True:
        time.sleep(0.5)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(('127.0.0.1', port))
        if result == 0:
            break
        sock.close()
    print("\n✨ [ระบบ] ComfyUI โหลดสำเร็จแล้ว! กำลังเชื่อมโยงลิ้งก์ภายนอก...\n")

    p = subprocess.Popen(["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    for line in p.stderr:
        l = line.decode()
        if "trycloudflare.com " in l:
            print("👉 เข้าใช้งานได้ที่ลิงก์นี้:", l[l.find("http"):].strip())

threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

print("🔥 5. กำลังเริ่มสตาร์ทระบบ ComfyUI Server...")
!python main.py --dont-print-server

🔄 1. กำลังตรวจสอบการเชื่อมต่อ Google Drive...
✅ Google Drive เชื่อมต่ออยู่แล้ว ข้ามขั้นตอนนี้ได้เลย
📌 โฟลเดอร์ทำงานปัจจุบัน: /content/drive/MyDrive/ComfyUI_Drive/ComfyUI
🔄 2. กำลังดึงอัปเดต ComfyUI ล่าสุด...
Already up to date.
Drive not mounted, so nothing to flush and unmount.
🔄 3. กำลังติดตั้งไลบรารี gguf และแพ็กเกจระบบเสริม...
🔥 5. กำลังเริ่มสตาร์ทระบบ ComfyUI Server...
[INFO] setup plugin alembic.autogenerate.schemas
[INFO] setup plugin alembic.autogenerate.tables
[INFO] setup plugin alembic.autogenerate.types
[INFO] setup plugin alembic.autogenerate.constraints
[INFO] setup plugin alembic.autogenerate.defaults
[INFO] setup plugin alembic.autogenerate.comments
[INFO] setup plugin alembic.ext.checkconstraint_byname
[INFO] Found comfy_kitchen backend hip: {'available': False, 'disabled': False, 'unavailable_reason': 'PyTorch ROCm/HIP runtime not available', 'capabilities': []}
[INFO] Found comfy_kitchen backend ascend: {'available': False, 'disabled': False, 'unavailable_reason': "t

```markdown
# ขั้นตอนที่ 1: เชื่อมต่อ Google Drive (Mount Drive)
```

In [ ]:
#@markdown <br><center><img src='https://upload.wikimedia.org/wikipedia/commons/thumb/d/da/Google_Drive_logo.png/600px-Google_Drive_logo.png' height="50" alt="Gdrive-logo"/></center>
#@markdown <center><h3>Mount Gdrive to /content/drive</h3></center><br>
MODE = "MOUNT" #@param ["MOUNT", "UNMOUNT"]
#Mount your Gdrive!
from google.colab import drive
drive.mount._DEBUG = False
if MODE == "MOUNT":
  drive.mount('/content/drive', force_remount=True)
elif MODE == "UNMOUNT":
  try:
    drive.flush_and_unmount()
  except ValueError:
    pass
  get_ipython().system_raw("rm -rf /root/.config/Google/DriveFS")

Mounted at /content/drive


In [ ]:
assert __import__('os').path.exists('/content/drive'), 'Google Drive is not mounted yet. Please run the cell above first.'
!ls -la /content/drive

total 28
drwx------ 6 root root 4096 Oct 25 17:13 .
drwxr-xr-x 1 root root 4096 Oct 25 17:13 ..
dr-x------ 4 root root 4096 Oct 25 17:13 .Encrypted
drwx------ 2 root root 4096 Oct 25 17:13 MyDrive
drwx------ 2 root root 4096 Oct 25 17:13 Othercomputers
dr-x------ 2 root root 4096 Oct 25 17:13 .shortcut-targets-by-id
drwx------ 5 root root 4096 Oct 25 17:13 .Trash-0


```markdown
# ขั้นตอนที่ 2: ติดตั้งและอัปเดต ComfyUI
```

In [ ]:
from pathlib import Path
import os

OPTIONS = {}

DRIVE_PATH = "/content/drive/MyDrive/ComfyUI_Drive"  # @param {type:"string"}
UPDATE_COMFY_UI = True  #@param {type:"boolean"}
WORKSPACE = '/content/ComfyUI'
OPTIONS['UPDATE_COMFY_UI'] = UPDATE_COMFY_UI

if DRIVE_PATH:
    WORKSPACE = DRIVE_PATH+"/ComfyUI"
    os.makedirs(DRIVE_PATH, exist_ok=True)
    %cd {DRIVE_PATH}

# Check if ComfyUI already exists in the drive path, if not clone it
if not os.path.exists(WORKSPACE):
    !echo -= Initial setup ComfyUI =-
    !git clone https://github.com/comfyanonymous/ComfyUI
else:
    print(f"ComfyUI already exists at: {WORKSPACE}")

%cd {WORKSPACE}

if OPTIONS['UPDATE_COMFY_UI']:
  !echo -= Updating ComfyUI =-
  !git pull

!echo -= Install dependencies =-
!pip install xformers!=0.0.18 -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu121 --extra-index-url https://download.pytorch.org/whl/cu118 --extra-index-url https://download.pytorch.org/whl/cu117


/content/drive/MyDrive/ComfyUI_Drive
ComfyUI already exists at: /content/drive/MyDrive/ComfyUI_Drive/ComfyUI
/content/drive/MyDrive/ComfyUI_Drive/ComfyUI
-= Updating ComfyUI =-
Already up to date.
-= Install dependencies =-
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu121, https://download.pytorch.org/whl/cu118, https://download.pytorch.org/whl/cu117


```markdown
# ขั้นตอนที่ 3: ดาวน์โหลดโมเดล (Models Download)
```

## Download MODELS

In [17]:
#@title 📥 Multi-Category Model Downloader (Civitai & Hugging Face)
import os
import subprocess
from google.colab import userdata

# 1. Configuration UI
MODEL_TYPE = "checkpoints" #@param ["checkpoints", "loras", "vae", "controlnet", "upscale_models", "embeddings", "unet", "clip"]
DOWNLOAD_URL = "https://civitai.com/api/download/models/2071650" #@param {type:"string"}
CUSTOM_FILENAME = "cyberrealisticPony_v127Alt.safetensors" #@param {type:"string"}

# Get Civitai Token if available
try:
  CIVITAI_API_TOKEN = userdata.get('CIVITAI_API_TOKEN')
except:
  CIVITAI_API_TOKEN = ""

# Ensure aria2 is installed for fast downloading
!apt-get -y install -qq aria2

# Set correct destination folder inside the active Workspace
try:
    base_workspace = WORKSPACE
except NameError:
    base_workspace = "/content/drive/MyDrive/ComfyUI_Drive/ComfyUI"

dest_dir = f"{base_workspace}/models/{MODEL_TYPE}"
os.makedirs(dest_dir, exist_ok=True)
print(f"🎯 ปลายทางเก็บโมเดล: {dest_dir}")

# Build URL and command
final_url = DOWNLOAD_URL
is_civitai = 'civitai.com' in DOWNLOAD_URL

if is_civitai:
    if CIVITAI_API_TOKEN.strip():
        separator = "&" if "?" in DOWNLOAD_URL else "?"
        final_url = f"{DOWNLOAD_URL}{separator}token={CIVITAI_API_TOKEN}"
    else:
        print("⚠️ คำเตือน: ตรวจพบลิงก์จาก Civitai แต่ไม่มีการกรอก API Token หากดาวน์โหลดล้มเหลว กรุณาตั้งค่าบานหน้าต่างด้านซ้าย (🔑 Secrets) โดยใช้ชื่อแปรว่า 'CIVITAI_API_TOKEN'")

out_filename = f"-o \"{CUSTOM_FILENAME}\"" if CUSTOM_FILENAME.strip() else ""

# Execute download
%cd {dest_dir}
print("🚀 กำลังเริ่มกระบวนการดาวน์โหลด...")

result = subprocess.run(f'aria2c --console-log-level=error -c -x 16 -s 16 -k 1M {out_filename} "{final_url}"', shell=True)

if result.returncode != 0:
    print("\n❌ ไม่สามารถดาวน์โหลดไฟล์ได้เนื่องจากข้อผิดพลาดบางประการ")
    if is_civitai and not CIVITAI_API_TOKEN.strip():
        print("💡 คำแนะนำ: โมเดล Civitai ตัวนี้จำเป็นต้องใช้สิทธิ์เข้าถึง (API Token) กรุณานำ Token มาเพิ่มในเมนูรูปกุญแจด้านข้างซ้าย แล้วเปิดสิทธิ์ให้ Notebook อ่านข้อมูล")
    else:
        print("💡 ระบบกำลังทดลองดาวน์โหลดสำรองด้วยคำสั่ง wget...")
        wget_filename = f"-O \"{CUSTOM_FILENAME}\"" if CUSTOM_FILENAME.strip() else ""
        !wget -c {wget_filename} "{DOWNLOAD_URL}"
else:
    print("🎉 ดาวน์โหลดโมเดลสำเร็จเรียบร้อยแล้ว!")

%cd {base_workspace}

🎯 ปลายทางเก็บโมเดล: /content/drive/MyDrive/ComfyUI_Drive/ComfyUI/models/checkpoints
⚠️ คำเตือน: ตรวจพบลิงก์จาก Civitai แต่ไม่มีการกรอก API Token หากดาวน์โหลดล้มเหลว กรุณาตั้งค่าบานหน้าต่างด้านซ้าย (🔑 Secrets) โดยใช้ชื่อแปรว่า 'CIVITAI_API_TOKEN'
/content/drive/MyDrive/ComfyUI_Drive/ComfyUI/models/checkpoints
🚀 กำลังเริ่มกระบวนการดาวน์โหลด...

❌ ไม่สามารถดาวน์โหลดไฟล์ได้เนื่องจากข้อผิดพลาดบางประการ
💡 คำแนะนำ: โมเดล Civitai ตัวนี้จำเป็นต้องใช้สิทธิ์เข้าถึง (API Token) กรุณานำ Token มาเพิ่มในเมนูรูปกุญแจด้านข้างซ้าย แล้วเปิดสิทธิ์ให้ Notebook อ่านข้อมูล
/content/drive/MyDrive/ComfyUI_Drive/ComfyUI


## OTHERS

In [ ]:
# SD1.5
!wget -c https://huggingface.co/runwayml/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.ckpt -P ./models/checkpoints/

# Some SD1.5 anime style
!wget -c https://huggingface.co/WarriorMama777/OrangeMixs/resolve/main/Models/AbyssOrangeMix2/AbyssOrangeMix2_hard.safetensors -P ./models/checkpoints/

# VAE
!wget -c https://huggingface.co/stabilityai/sd-vae-ft-mse-original/resolve/main/vae-ft-mse-840000-ema-pruned.safetensors -P ./models/vae/

## LIST MODELS

In [ ]:
!ls -al ./models/checkpoints/

total 6775444
drwxr-xr-x  2 root root       4096 Aug 30 23:40 .
drwxr-xr-x 22 root root       4096 Aug 30 23:39 ..
-rw-r--r--  1 root root 6938040682 Aug 30 23:40 cyberrealisticPony_v127Alt.safetensors
-rw-r--r--  1 root root          0 Aug 30 23:39 put_checkpoints_here


```markdown
# ขั้นตอนที่ 4: ดาวน์โหลดส่วนเสริมและเครื่องมือเชื่อมต่อ (Prerequisites)
```

```markdown
### ติดตั้งแพ็กเกจเสริมและ GGUF Python Package ป้องกัน Error
```

In [1]:
# ติดตั้ง gguf เพื่อป้องกัน ModuleNotFoundError
!pip install gguf

# ติดตั้ง Cloudflared สำหรับการทำ Tunnel
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

Selecting previously unselected package cloudflared.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.10.3) ...
Setting up cloudflared (2026.10.3) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [ ]:
!python main.py --dont-print-server

[INFO] setup plugin alembic.autogenerate.schemas
[INFO] setup plugin alembic.autogenerate.tables
[INFO] setup plugin alembic.autogenerate.types
[INFO] setup plugin alembic.autogenerate.constraints
[INFO] setup plugin alembic.autogenerate.defaults
[INFO] setup plugin alembic.autogenerate.comments
[INFO] setup plugin alembic.ext.checkconstraint_byname
[INFO] Found comfy_kitchen backend cuda: {'available': True, 'disabled': False, 'unavailable_reason': None, 'capabilities': ['adaln', 'apply_rope', 'apply_rope1', 'apply_rope1_', 'apply_rope_', 'apply_rope_split_half', 'apply_rope_split_half1', 'apply_rope_split_half1_', 'apply_rope_split_half_', 'closest_point_on_mesh', 'connected_components', 'convrot_w4a4_linear', 'delaunay3d', 'dequantize_convrot_w4a4_weight', 'dequantize_int8_convrot_weight', 'dequantize_int8_convrot_weight_dtype', 'dequantize_int8_simple', 'dequantize_int8_simple_dtype', 'dequantize_nvfp4', 'dequantize_per_tensor_fp8', 'dequantize_w4a8_int8_weight', 'edge_collapse_che

```markdown
# ขั้นตอนที่ 5: เริ่มต้นใช้งาน ComfyUI & ลิงก์เข้าใช้งาน (Start Server)
```

```markdown
### เลือกช่องทางการเปิดเซิร์ฟเวอร์ด้านล่างนี้ (แนะนำใช้ Cloudflare Tunnel)
```

In [2]:
# ติดตั้ง localtunnel ผ่าน npm สำหรับทางเลือกสำรอง
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙
added 22 packages in 2s
⠙
⠙3 packages are looking for funding
⠙  run `npm fund` for details
⠙

## CF Tunnel

In [14]:
# เปิดใช้งานผ่าน Cloudflare Tunnel (วิธีหลัก)
import subprocess
import threading
import time
import socket
import os

# ดึงตำแหน่งพื้นที่ทำงานปัจจุบันจากตัวแปรส่วนกลางที่เชื่อมต่อไว้สำเร็จ
try:
    workspace_path = WORKSPACE
except NameError:
    workspace_path = "/content/drive/MyDrive/ComfyUI_Drive/ComfyUI"

# ตรวจสอบและสลับตำแหน่งพาร์ทสำรองหากพาร์ทหลักยังไม่ถูกดึงขึ้นมา
if not os.path.exists(workspace_path):
    fallback_paths = [
        "/content/drive/MyDrive/ComfyUI_Drive/ComfyUI",
        "/content/drive/MyDrive/ComfyUI",
        "/content/ComfyUI"
    ]
    for path in fallback_paths:
        if os.path.exists(path):
            workspace_path = path
            break

def iframe_thread(port):
    while True:
        time.sleep(0.5)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(('127.0.0.1', port))
        if result == 0:
            break
        sock.close()
    print("\n[ระบบ] ComfyUI โหลดเสร็จสิ้น กำลังสร้างลิงก์เข้าใช้งานผ่าน Cloudflared...\n")

    p = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:{}".format(port)], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    for line in p.stderr:
        l = line.decode()
        if "trycloudflare.com " in l:
            print("👉 คลิกที่ลิงก์นี้เพื่อเข้าใช้งาน ComfyUI:", l[l.find("http"):].strip())

# เริ่มทำงานตรวจสอบพอร์ตเบื้องหลัง
threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

# ย้ายตำแหน่งเพื่อรันเซิร์ฟเวอร์ด้วยตำแหน่งที่ตรวจสอบพบจริง
try:
    print(f"📌 กำลังเปลี่ยนโฟลเดอร์ทำงานไปที่: {workspace_path}")
    os.chdir(workspace_path)
    !python main.py --dont-print-server
except Exception as e:
    print("❌ เกิดข้อผิดพลาดในการเข้าสู่ไดเรกทอรี ComfyUI: ", str(e))
    print("กรุณาลองตรวจสอบการ Mount Google Drive ในขั้นตอนแรกใหม่อีกครั้งเพื่อรีเฟรชสิทธิ์การใช้งาน")

📌 กำลังเปลี่ยนโฟลเดอร์ทำงานไปที่: /content/drive/MyDrive/ComfyUI_Drive/ComfyUI
❌ เกิดข้อผิดพลาดในการเข้าสู่ไดเรกทอรี ComfyUI:  [Errno 2] No such file or directory: '/content/drive/MyDrive/ComfyUI_Drive/ComfyUI'
กำลังตรวจสอบโฟลเดอร์ที่เข้าถึงได้บนเครื่องของคุณเพื่อการแก้ไขปัญหา...


## localtunnel

In [ ]:
# localtunnel
!npm install -g localtunnel

import subprocess
import threading
import time
import socket
import urllib.request
import os

def iframe_thread(port):
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      result = sock.connect_ex(('127.0.0.1', port))
      if result == 0:
        break
      sock.close()
  print("\nComfyUI finished loading, trying to launch localtunnel (if it gets stuck here localtunnel is having issues)\n")

  print("The password/enpoint ip for localtunnel is:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n"))
  p = subprocess.Popen(["lt", "--port", "{}".format(port)], stdout=subprocess.PIPE)
  for line in p.stdout:
    print(line.decode(), end='')

threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

# Ensure we navigate to the valid WORKSPACE path on Google Drive before running
%cd {WORKSPACE}
!python main.py --dont-print-server


⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸
changed 22 packages in 1s
⠸
⠸3 packages are looking for funding
⠸  run `npm fund` for details
⠸/content/drive/MyDrive/ComfyUI_Drive/ComfyUI
[INFO] setup plugin alembic.autogenerate.schemas
[INFO] setup plugin alembic.autogenerate.tables
[INFO] setup plugin alembic.autogenerate.types
[INFO] setup plugin alembic.autogenerate.constraints
[INFO] setup plugin alembic.autogenerate.defaults
[INFO] setup plugin alembic.autogenerate.comments
[INFO] setup plugin alembic.ext.checkconstraint_byname
[INFO] Found comfy_kitchen backend hip: {'available': False, 'disabled': False, 'unavailable_reason': 'PyTorch ROCm/HIP runtime not available', 'capabilities': []}
[INFO] Found comfy_kitchen backend cuda: {'available': True, 'disabled': False, 'unavailable_reason': None, 'capabilities': ['adaln', 'apply_rope', 'apply_rope1', 'apply_rope1_', 'apply_rope_', 'apply_rope_split_half', 'apply_rope_split_half1', 'apply_rope_split_half1_', 'apply_rope_split_half_', 'closest_point_on_mesh', 'connec